# Lesson 7B: Anomaly Detection - Practical

## Credit Card Fraud Detection

**Case Study**: A credit card company processes 10 million transactions daily. Only 0.1% are fraudulent. How do we find them?

**Challenge**: Extreme class imbalance → unsupervised anomaly detection is ideal.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

np.random.seed(42)
print("✅ Loaded!")

## Step 1: Simulate Transaction Data

Features:
- **Amount**: Transaction amount ($)
- **Frequency**: Transactions per day
- **Normal**: Small amounts, consistent frequency
- **Fraud**: Large amounts, unusual patterns

In [ ]:
# Simulate credit card transactions
normal_transactions = np.random.randn(1000, 2) * [20, 100] + [50, 500]
fraud_transactions = np.random.randn(50, 2) * [50, 200] + [200, 2000]
X = np.vstack([normal_transactions, fraud_transactions])
y_true = np.hstack([np.zeros(1000), np.ones(50)])

print(f"Total transactions: {len(X)}")
print(f"Normal: {sum(y_true == 0)} ({sum(y_true == 0)/len(X)*100:.1f}%)")
print(f"Fraud: {sum(y_true == 1)} ({sum(y_true == 1)/len(X)*100:.1f}%)")

# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("✅ Data prepared!")

## Step 2: Compare Anomaly Detection Methods

We'll test all three methods and evaluate their performance.

In [ ]:
# Train all three models
models = {
    'Isolation Forest': IsolationForest(contamination=0.05, random_state=42),
    'LOF': LocalOutlierFactor(contamination=0.05, novelty=False),
    'One-Class SVM': OneClassSVM(nu=0.05, gamma='auto')
}

results = {}

for name, model in models.items():
    if name == 'LOF':
        y_pred = model.fit_predict(X_scaled)
    else:
        y_pred = model.fit(X_scaled).predict(X_scaled)
    
    # Convert to binary (1 = anomaly)
    y_pred_binary = (y_pred == -1).astype(int)
    results[name] = y_pred_binary
    
    print(f"
{name}:")
    print(classification_report(y_true, y_pred_binary, target_names=['Normal', 'Fraud'], 
                                zero_division=0))

print("✅ All models evaluated!")

## Step 3: Parameter Tuning - Contamination

The **contamination** parameter sets expected fraction of anomalies. Let's tune it!

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

# Test different contamination values
contaminations = [0.01, 0.03, 0.05, 0.07, 0.10, 0.15]
precisions, recalls, f1s = [], [], []

for cont in contaminations:
    iso = IsolationForest(contamination=cont, random_state=42)
    y_pred = iso.fit_predict(X_scaled)
    y_pred_binary = (y_pred == -1).astype(int)
    
    precisions.append(precision_score(y_true, y_pred_binary, zero_division=0))
    recalls.append(recall_score(y_true, y_pred_binary))
    f1s.append(f1_score(y_true, y_pred_binary, zero_division=0))

# Plot
plt.figure(figsize=(10, 6))
plt.plot(contaminations, precisions, 'bo-', label='Precision', linewidth=2)
plt.plot(contaminations, recalls, 'ro-', label='Recall', linewidth=2)
plt.plot(contaminations, f1s, 'go-', label='F1 Score', linewidth=2)
plt.xlabel('Contamination Parameter', fontsize=12)
plt.ylabel('Score', fontsize=12)
plt.title('Isolation Forest Parameter Tuning', fontweight='bold', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.show()

best_idx = np.argmax(f1s)
print(f"Best contamination: {contaminations[best_idx]}")
print(f"Best F1 score: {f1s[best_idx]:.3f}")
print("✅ Parameter tuning complete!")

## Step 4: Anomaly Scores

Most models provide anomaly scores (not just binary predictions). Higher scores = more anomalous.

In [ ]:
# Get anomaly scores
iso = IsolationForest(contamination=0.05, random_state=42)
iso.fit(X_scaled)

# Anomaly scores (more negative = more anomalous)
scores = iso.score_samples(X_scaled)

# Visualize scores
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.hist(scores[y_true == 0], bins=50, alpha=0.7, label='Normal', color='blue')
plt.hist(scores[y_true == 1], bins=50, alpha=0.7, label='Fraud', color='red')
plt.xlabel('Anomaly Score (more negative = anomalous)', fontsize=11)
plt.ylabel('Count', fontsize=11)
plt.title('Anomaly Score Distribution', fontweight='bold', fontsize=13)
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
# Scatter plot with score coloring
scatter = plt.scatter(X[:, 0], X[:, 1], c=scores, cmap='RdYlBu', s=30, alpha=0.7)
plt.colorbar(scatter, label='Anomaly Score')
plt.scatter(X[y_true == 1, 0], X[y_true == 1, 1], 
           facecolors='none', edgecolors='red', s=100, linewidths=2, label='True Fraud')
plt.xlabel('Transaction Amount ($)', fontsize=11)
plt.ylabel('Frequency', fontsize=11)
plt.title('Anomaly Scores Visualization', fontweight='bold', fontsize=13)
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✅ Anomaly scores visualized!")

## Step 5: Visualize Results

In [ ]:
# Final comparison visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# True labels
ax = axes[0, 0]
ax.scatter(X[y_true==0, 0], X[y_true==0, 1], c='blue', s=20, alpha=0.5, label='Normal')
ax.scatter(X[y_true==1, 0], X[y_true==1, 1], c='red', s=50, marker='x', 
          linewidths=2, label='Fraud')
ax.set_xlabel('Amount ($)')
ax.set_ylabel('Frequency')
ax.set_title('True Labels', fontweight='bold', fontsize=13)
ax.legend()
ax.grid(True, alpha=0.3)

# Model predictions
for idx, (name, y_pred) in enumerate(results.items()):
    ax = axes.flatten()[idx + 1]
    normal_mask = y_pred == 0
    fraud_mask = y_pred == 1
    
    ax.scatter(X[normal_mask, 0], X[normal_mask, 1], c='blue', s=20, alpha=0.5, label='Normal')
    ax.scatter(X[fraud_mask, 0], X[fraud_mask, 1], c='red', s=50, marker='x', 
              linewidths=2, label='Detected Fraud')
    ax.set_xlabel('Amount ($)')
    ax.set_ylabel('Frequency')
    ax.set_title(f'{name}', fontweight='bold', fontsize=13)
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Add accuracy
    from sklearn.metrics import accuracy_score
    acc = accuracy_score(y_true, y_pred)
    ax.text(0.02, 0.98, f'Accuracy: {acc:.2%}', transform=ax.transAxes, 
           verticalalignment='top', fontsize=10, bbox=dict(boxstyle='round', 
           facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()
print("✅ Fraud detection complete!")

## Production Best Practices

1. **Handle imbalance**: Use contamination parameter based on historical fraud rate
2. **Monitor scores**: Track anomaly score distributions over time
3. **Human review**: High-score anomalies should be manually verified
4. **Retrain regularly**: Fraud patterns evolve, retrain monthly
5. **Feature engineering**: Add temporal, geographic, behavioral features
6. **Ensemble**: Combine multiple methods for better detection

**Real-world deployment**:
```python
# Example production pipeline
def detect_fraud(transaction):
    features = extract_features(transaction)
    score = model.score_samples([features])[0]
    
    if score < threshold_high_risk:
        return 'BLOCK', score
    elif score < threshold_review:
        return 'REVIEW', score
    else:
        return 'APPROVE', score
```